# 01 — Data Exploration

This notebook demonstrates how to load, inspect, and evaluate datasets
using the `scenario-reasoner-lm` data infrastructure.

## What you'll learn
- Creating `ScenarioDataset` and `ReasoningTraceDataset` instances
- Using `DatasetEvaluator` to understand your data
- Wrapping HuggingFace datasets with `HFDatasetWrapper`
- Creating PyTorch DataLoaders

In [ ]:
import sys
sys.path.insert(0, '..')  # Make src importable from notebooks/

from src.data.custom_datasets import ScenarioDataset, ReasoningTraceDataset
from src.data.evaluators import DatasetEvaluator

## 1. Create a ScenarioDataset

In [ ]:
scenarios = [
    "A hospital has 3 doctors and 10 patients. Two doctors call in sick. How many patients per doctor?",
    "A train leaves at 9am travelling at 60mph. Another leaves at 10am at 80mph. When does the second overtake the first?",
    "A farmer has 17 sheep. All but 9 die. How many sheep are left?",
]
labels = ["10", "1pm", "9"]

dataset = ScenarioDataset(scenarios=scenarios, labels=labels, split="train")
print(f"Dataset size: {len(dataset)}")
print(f"Sample 0: {dataset[0]}")

## 2. Create a ReasoningTraceDataset

In [ ]:
traces = [
    "Step 1: 3 - 2 = 1 doctor available. Step 2: 10 / 1 = 10. Therefore: 10 patients per doctor.",
    "Step 1: At 10am the first train has travelled 60 miles. Step 2: Relative speed = 20mph. Step 3: 60/20 = 3 hours. Therefore: 1pm.",
    "'All but 9' means 9 remain. Therefore: 9 sheep.",
]
outputs = ["10", "1pm", "9"]

trace_dataset = ReasoningTraceDataset(scenarios, traces, outputs)
sample = trace_dataset[1]
print("Input:", sample["input"])
print("Trace:", sample["reasoning_trace"])
print("Output:", sample["output"])

## 3. Evaluate the Dataset

In [ ]:
import json
evaluator = DatasetEvaluator(dataset)
report = evaluator.evaluate()
print(json.dumps(report, indent=2))

## 4. Create a DataLoader

In [ ]:
loader = dataset.get_dataloader(batch_size=2, shuffle=False)
for batch in loader:
    print("Batch keys:", list(batch.keys()))
    print("Inputs:", batch["input"])
    break